# Assignment 3 - NYC Taxi Big Data Processing

In [1]:
import dask
from dask_jobqueue import SLURMCluster
from dask.distributed import Client
import dask.dataframe as dd
from pathlib import Path
import os
import pandas as pd
import geopandas as gpd

## Cluster setup (SLURM + Dask)

In [2]:
# Verify that Dask is installed
print("Dask version:", dask.__version__)

Dask version: 2024.5.0


In [3]:
cluster = SLURMCluster(
    queue="all",
    cores=1, # lower number of cores per worker to reduce memory usage
    memory="16GB",
    walltime="02:00:00",
    job_extra=["--output=/d/hpc/home/sm79111/bigdata/logs/slurm-%j.out"]
)

cluster.scale(8)  # Scale to 8 workers

client = Client(cluster)
client

/d/hpc/home/sm79111/.conda/envs/bd39/lib/python3.9/site-packages/dask_jobqueue/core.py:266: FutureWarning: job_extra has been renamed to job_extra_directives. You are still using it (even if only set to []; please also check config files). If you did not set job_extra_directives yet, job_extra will be respected for now, but it will be removed in a future release. If you already set job_extra_directives, job_extra is ignored and you can remove it.
  warnings.warn(warn, FutureWarning)
/d/hpc/home/sm79111/.conda/envs/bd39/lib/python3.9/site-packages/distributed/node.py:182: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 46787 instead
  warnings.warn(
/d/hpc/home/sm79111/.conda/envs/bd39/lib/python3.9/site-packages/dask_jobqueue/core.py:266: FutureWarning: job_extra has been renamed to job_extra_directives. You are still using it (even if only set to []; please also check config files). If you did not set job_extra_dire

Connection method: Cluster object,Cluster type: dask_jobqueue.SLURMCluster
Dashboard: http://153.5.72.244:46787/status,
Dashboard: http://153.5.72.244:46787/status,Workers: 0
Total threads: 0,Total memory: 0 B
Comm: tcp://153.5.72.244:36671,Workers: 0
Dashboard: http://153.5.72.244:46787/status,Total threads: 0
Started: Just now,Total memory: 0 B


## Dataset Overview
We focus on the **yellow taxi dataset**, which is the largest and most commonly used.

In [4]:
DATA_DIR = Path("/d/hpc/projects/FRI/bigdata/data/Taxi")

files = sorted(DATA_DIR.rglob("yellow_tripdata_*.parquet"))

print("Total yellow taxi parquet files:", len(files))

for f in files[:10]:
    print(f.name)

Total yellow taxi parquet files: 193
yellow_tripdata_2009-01.parquet
yellow_tripdata_2009-02.parquet
yellow_tripdata_2009-03.parquet
yellow_tripdata_2009-04.parquet
yellow_tripdata_2009-05.parquet
yellow_tripdata_2009-06.parquet
yellow_tripdata_2009-07.parquet
yellow_tripdata_2009-08.parquet
yellow_tripdata_2009-09.parquet
yellow_tripdata_2009-10.parquet


### Dataset Organization by Year
To simplify processing we group the parquet files by year.
Each year contains monthly files.

In [5]:
files_by_year = {}

for f in files:
    year = f.stem.split("_")[-1][:4]
    files_by_year.setdefault(year, []).append(str(f))

print("Available years:")
print(sorted(files_by_year.keys()))

Available years:
['2009', '2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024', '2025']


### Schema Inspection and Harmonization
Taxi datasets evolve over time.

Column names and data types may change between years. Before combining datasets we inspect schemas to detect inconsistencies.

In [6]:
selected_years = ["2019", "2020", "2021", "2022", "2023"]

for year in selected_years:
    df = dd.read_parquet(files_by_year[year])
    
    print("\n===== YEAR", year, "=====")
    print("Number of columns:", len(df.columns))
    print(df.dtypes.astype(str))


===== YEAR 2019 =====
Number of columns: 19
VendorID                          int64
tpep_pickup_datetime     datetime64[us]
tpep_dropoff_datetime    datetime64[us]
passenger_count                 float64
trip_distance                   float64
RatecodeID                      float64
store_and_fwd_flag               string
PULocationID                      int64
DOLocationID                      int64
payment_type                      int64
fare_amount                     float64
extra                           float64
mta_tax                         float64
tip_amount                      float64
tolls_amount                    float64
improvement_surcharge           float64
total_amount                    float64
congestion_surcharge            float64
airport_fee                      object
dtype: object

===== YEAR 2020 =====
Number of columns: 19
VendorID                          int64
tpep_pickup_datetime     datetime64[us]
tpep_dropoff_datetime    datetime64[us]
passenger_count 

In [7]:
schemas = {}

for year in selected_years:
    df = dd.read_parquet(files_by_year[year])
    schemas[year] = set(df.columns)

for year, cols in schemas.items():
    print(year, len(cols))

2019 19
2020 19
2021 19
2022 19
2023 19


## Data Ingestion Tasks

1. CSV for 5 years
2. CSV + HDF5 for 1 year
3. Optimized Parquet partitions

In [8]:
OUTPUT_DIR = Path("/d/hpc/home/sm79111/bigdata/output")

CSV_DIR = OUTPUT_DIR / "csv"
HDF_DIR = OUTPUT_DIR / "hdf"
PARQUET_DIR = OUTPUT_DIR / "parquet"

CSV_DIR.mkdir(parents=True, exist_ok=True)
HDF_DIR.mkdir(parents=True, exist_ok=True)
PARQUET_DIR.mkdir(parents=True, exist_ok=True)

### Schema Normalization

Taxi datasets evolve over time. Some columns appear only in later years and data types may change. To ensure consistency across datasets we define a
target schema and normalize each partition accordingly.

In [9]:
target_columns = [
    "VendorID",
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "passenger_count",
    "trip_distance",
    "RatecodeID",
    "store_and_fwd_flag",
    "PULocationID",
    "DOLocationID",
    "payment_type",
    "fare_amount",
    "extra",
    "mta_tax",
    "tip_amount",
    "tolls_amount",
    "improvement_surcharge",
    "total_amount",
    "congestion_surcharge",
    "airport_fee"
]

def normalize_partition(pdf):

    pdf = pdf.copy()

    # add missing columns
    for col in target_columns:
        if col not in pdf.columns:
            pdf[col] = pd.NA

    # reorder columns
    pdf = pdf[target_columns]

    numeric_cols = [
        "VendorID","passenger_count","trip_distance","RatecodeID",
        "PULocationID","DOLocationID","payment_type",
        "fare_amount","extra","mta_tax","tip_amount",
        "tolls_amount","improvement_surcharge","total_amount",
        "congestion_surcharge","airport_fee"
    ]

    for col in numeric_cols:
        pdf[col] = pd.to_numeric(pdf[col], errors="coerce")

    pdf["store_and_fwd_flag"] = pdf["store_and_fwd_flag"].astype("string")

    pdf["tpep_pickup_datetime"] = pd.to_datetime(
        pdf["tpep_pickup_datetime"], errors="coerce"
    )

    pdf["tpep_dropoff_datetime"] = pd.to_datetime(
        pdf["tpep_dropoff_datetime"], errors="coerce"
    )

    return pdf

In [11]:
# Define metadata 
meta = pd.DataFrame({
    "VendorID": pd.Series(dtype="float64"),
    "tpep_pickup_datetime": pd.Series(dtype="datetime64[ns]"),
    "tpep_dropoff_datetime": pd.Series(dtype="datetime64[ns]"),
    "passenger_count": pd.Series(dtype="float64"),
    "trip_distance": pd.Series(dtype="float64"),
    "RatecodeID": pd.Series(dtype="float64"),
    "store_and_fwd_flag": pd.Series(dtype="string"),
    "PULocationID": pd.Series(dtype="float64"),
    "DOLocationID": pd.Series(dtype="float64"),
    "payment_type": pd.Series(dtype="float64"),
    "fare_amount": pd.Series(dtype="float64"),
    "extra": pd.Series(dtype="float64"),
    "mta_tax": pd.Series(dtype="float64"),
    "tip_amount": pd.Series(dtype="float64"),
    "tolls_amount": pd.Series(dtype="float64"),
    "improvement_surcharge": pd.Series(dtype="float64"),
    "total_amount": pd.Series(dtype="float64"),
    "congestion_surcharge": pd.Series(dtype="float64"),
    "airport_fee": pd.Series(dtype="float64"),
    "year": pd.Series(dtype="int64")
})

# Normalize data for selected years and store in a dictionary
normalized_dfs = {}

for year in selected_years:

    print("Processing year:", year)

    yearly_parts = []

    # Read and normalize each parquet file separately to handle schema drift
    # across monthly files (e.g., airport_fee missing in some 2023 files).
    for file_path in files_by_year[year]:
        part_df = dd.read_parquet(
            file_path,
            engine="pyarrow"
        )

        for col in target_columns:
            if col not in part_df.columns:
                part_df[col] = None

        part_df = part_df.map_partitions(
            normalize_partition,
            meta=meta.drop(columns=["year"])
        )

        yearly_parts.append(part_df)

    df = dd.concat(yearly_parts, interleave_partitions=True)
    df = df.assign(year=int(year))

    # Persist the normalized DataFrame to avoid re-computation in later steps
    normalized_dfs[year] = df.persist()

Processing year: 2019
Processing year: 2020
Processing year: 2021
Processing year: 2022
Processing year: 2023


### Export Five Years of Data to CSV
CSV is a common but inefficient storage format. We export five years of
data to CSV for comparison with other formats.

In [13]:
for year in selected_years:

    out_dir = CSV_DIR / f"csv_{year}"
    out_dir.mkdir(parents=True, exist_ok=True)

    df = normalized_dfs[year]

    print(year, "partitions:", df.npartitions)

    df.to_csv(
        str(out_dir / "part-*.csv"),
        index=False,
        compute_kwargs={"retries": 2}
    )

    print(f"{year} CSV exported")

2019 partitions: 12


2019 CSV exported
2020 partitions: 12
2020 CSV exported
2021 partitions: 12
2021 CSV exported
2022 partitions: 12
2022 CSV exported
2023 partitions: 12
2023 CSV exported


### Export for 1 year in both _HDF5_ and _CSV_

In [12]:
one_year = "2023"
one_year_df = normalized_dfs[one_year].persist()

one_year_csv_dir = OUTPUT_DIR / f"one_year_csv_{one_year}"
one_year_csv_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
# Export CSV
one_year_df.to_csv(
    str(one_year_csv_dir / "part-*.csv"),
    index=False
)

In [12]:
df_2023 = dd.read_csv(str(one_year_csv_dir / "part-*.csv"), assume_missing=True)

for i in range(df_2023.npartitions):
    df_2023.get_partition(i).compute().to_hdf(
        HDF_DIR / "2023.h5",
        key="taxi",
        mode="a",
        format="table",
        append=True
    )

In [13]:
pd.read_hdf(HDF_DIR / "2023.h5", key="taxi").head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee,year
0,2.0,2023-01-01 00:32:10,2023-01-01 00:40:36,1.0,0.97,1.0,N,161.0,141.0,2.0,9.3,1.00,0.5,0.00,0.0,1.0,14.30,2.5,0.00,2023.0
1,2.0,2023-01-01 00:55:08,2023-01-01 01:01:27,1.0,1.10,1.0,N,43.0,237.0,1.0,7.9,1.00,0.5,4.00,0.0,1.0,16.90,2.5,0.00,2023.0
2,2.0,2023-01-01 00:25:04,2023-01-01 00:37:49,1.0,2.51,1.0,N,48.0,238.0,1.0,14.9,1.00,0.5,15.00,0.0,1.0,34.90,2.5,0.00,2023.0
3,1.0,2023-01-01 00:03:48,2023-01-01 00:13:25,0.0,1.90,1.0,N,138.0,7.0,1.0,12.1,7.25,0.5,0.00,0.0,1.0,20.85,0.0,1.25,2023.0
4,2.0,2023-01-01 00:10:29,2023-01-01 00:21:19,1.0,1.43,1.0,N,107.0,79.0,1.0,11.4,1.00,0.5,3.28,0.0,1.0,19.68,2.5,0.00,2023.0


### Optimized Parquet Creation

In [16]:
# Concatenate all years into a single Dask DataFrame for partitioned output
all_years_df = dd.concat([df for df in normalized_dfs.values()])

# Repartition by approximate size (256 MB per partition)
all_years_df = all_years_df.repartition(partition_size="256MB")

all_years_df.to_parquet(
    PARQUET_DIR / "optimized_parquet",
    engine="pyarrow",
    write_index=False,
    partition_on=["year"],
    row_group_size=100000,
    compute_kwargs={"retries": 2}
)

## Storage Size Comparison

In [11]:
def path_size_bytes(path: Path) -> int:
    path = Path(path)
    if path.is_file():
        return path.stat().st_size
    return sum(p.stat().st_size for p in path.rglob("*") if p.is_file())

def fmt_gb(num_bytes: int) -> float:
    return num_bytes / (1024 ** 3)

In [12]:
# 1) Compare the 5-year exports
original_parquet_size = sum(path_size_bytes(p) for year in selected_years for p in files_by_year[year])
csv_5yr_size = sum(path_size_bytes(CSV_DIR / f"csv_{year}") for year in selected_years)
parquet_5yr_size = path_size_bytes(PARQUET_DIR / "optimized_parquet")

five_year_compare = pd.DataFrame([
    {"format": "Original Parquet (2019-2023)", "size_gb": fmt_gb(original_parquet_size)},
    {"format": "CSV (2019-2023)", "size_gb": fmt_gb(csv_5yr_size)},
    {"format": "Parquet (2019-2023)", "size_gb": fmt_gb(parquet_5yr_size)},
])

five_year_compare["ratio_vs_csv"] = five_year_compare["size_gb"] / five_year_compare.loc[
    five_year_compare["format"] == "CSV (2019-2023)", "size_gb"
].iloc[0]

print("Five-year storage comparison:")
print(five_year_compare.sort_values("size_gb").to_string(index=False))

Five-year storage comparison:
                      format   size_gb  ratio_vs_csv
Original Parquet (2019-2023)  3.118586      0.142814
         Parquet (2019-2023)  4.005278      0.183419
             CSV (2019-2023) 21.836775      1.000000


In [15]:
# 2) Compare the 2023 dataset in all available formats
original_parquet_2023_size = sum(path_size_bytes(p) for p in files_by_year[one_year])
csv_2023_size = path_size_bytes(one_year_csv_dir)
hdf_2023_size = path_size_bytes(HDF_DIR / "2023.h5")
parquet_2023_size = path_size_bytes(PARQUET_DIR / "optimized_parquet" / f"year={one_year}")

one_year_compare = pd.DataFrame([
    {"format": "Original Parquet (2023)", "size_gb": fmt_gb(original_parquet_2023_size)},
    {"format": "CSV (2023)", "size_gb": fmt_gb(csv_2023_size)},
    {"format": "HDF5 (2023)", "size_gb": fmt_gb(hdf_2023_size)},
    {"format": "Parquet (2023)", "size_gb": fmt_gb(parquet_2023_size)},
])

one_year_compare["ratio_vs_csv"] = one_year_compare["size_gb"] / one_year_compare.loc[
    one_year_compare["format"] == "CSV (2023)", "size_gb"
].iloc[0]

print("\n2023 storage comparison:")
print(one_year_compare.sort_values("size_gb").to_string(index=False))


2023 storage comparison:
                 format  size_gb  ratio_vs_csv
Original Parquet (2023) 0.592082      0.154813
         Parquet (2023) 0.760043      0.198730
             CSV (2023) 3.824503      1.000000
            HDF5 (2023) 7.190747      1.880178


## Data Augmentation

In [14]:
taxi_df = dd.read_parquet(PARQUET_DIR / "optimized_parquet")
taxi_df.head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee,year
0,1.0,2019-01-01 00:46:40,2019-01-01 00:53:20,1.0,1.5,1.0,N,151.0,239.0,1.0,7.0,0.5,0.5,1.65,0.0,0.3,9.95,NaN,NaN,2019
1,1.0,2019-01-01 00:59:47,2019-01-01 01:18:59,1.0,2.6,1.0,N,239.0,246.0,1.0,14.0,0.5,0.5,1.00,0.0,0.3,16.30,NaN,NaN,2019
2,2.0,2018-12-21 13:48:30,2018-12-21 13:52:40,3.0,0.0,1.0,N,236.0,236.0,1.0,4.5,0.5,0.5,0.00,0.0,0.3,5.80,NaN,NaN,2019
3,2.0,2018-11-28 15:52:25,2018-11-28 15:55:45,5.0,0.0,1.0,N,193.0,193.0,2.0,3.5,0.5,0.5,0.00,0.0,0.3,7.55,NaN,NaN,2019
4,2.0,2018-11-28 15:56:57,2018-11-28 15:58:33,5.0,0.0,2.0,N,193.0,193.0,2.0,52.0,0.0,0.5,0.00,0.0,0.3,55.55,NaN,NaN,2019


In [15]:
print("Taxi partitions:", taxi_df.npartitions)

Taxi partitions: 206


In [16]:
# Create an hourly timestamp column for joining
taxi_df = taxi_df.assign(
    pickup_time_hour=taxi_df["tpep_pickup_datetime"].dt.floor("h")
)

### Weather information (based on pickup/dropoff date-time)

Dataset: NYC Weather Data (Weather data of New York City from 2016 to Oct 25, 2022. This was downloaded from the Open-Meteo website.)

In [17]:
PARQUET_OUTPUT = OUTPUT_DIR / "augmented_weather_parquet"

In [24]:
weather = pd.read_csv("/d/hpc/home/sm79111/bigdata/NYC_Weather_2016_2022.csv")
weather.head()

,time,temperature_2m (°C),precipitation (mm),rain (mm),cloudcover (%),cloudcover_low (%),cloudcover_mid (%),cloudcover_high (%),windspeed_10m (km/h),winddirection_10m (°)
0,2016-01-01T00:00,7.6,0.0,0.0,69.0,53.0,0.0,72.0,10.0,296.0
1,2016-01-01T01:00,7.5,0.0,0.0,20.0,4.0,0.0,56.0,9.8,287.0
2,2016-01-01T02:00,7.1,0.0,0.0,32.0,3.0,0.0,99.0,9.7,285.0
3,2016-01-01T03:00,6.6,0.0,0.0,35.0,5.0,0.0,100.0,9.2,281.0
4,2016-01-01T04:00,6.3,0.0,0.0,34.0,4.0,0.0,100.0,9.1,279.0


In [25]:
weather["time"] = pd.to_datetime(weather["time"], errors="coerce")
weather = weather.dropna(subset=["time", "temperature_2m (°C)", "precipitation (mm)"])

In [27]:
weather = weather.rename(columns={
    "temperature_2m (°C)": "temperature_c",
    "precipitation (mm)": "precipitation_mm",
    "rain (mm)": "rain_mm",
    "cloudcover (%)": "cloudcover_pct",
    "windspeed_10m (km/h)": "windspeed_kmh",
})

In [28]:
weather["pickup_time_hour"] = weather["time"].dt.floor("h")

In [29]:
weather_small = weather[
    [
        "pickup_time_hour",
        "temperature_c",
        "precipitation_mm",
        "rain_mm",
        "cloudcover_pct",
        "windspeed_kmh",
    ]
].drop_duplicates()

In [30]:
weather_small["pickup_time_hour"] = weather_small["pickup_time_hour"].astype("datetime64[ns]")

In [31]:
# Convert weather data to Dask
weather_dd = dd.from_pandas(weather_small, npartitions=4)

In [39]:
# Filter Taxi Data to Weather Time Range, preventing unmatched timestamps during join
weather_min = weather_small["pickup_time_hour"].min()
weather_max = weather_small["pickup_time_hour"].max()

taxi_df = taxi_df[
    (taxi_df["tpep_pickup_datetime"] >= weather_min) &
    (taxi_df["tpep_pickup_datetime"] <= weather_max)
]

In [40]:
# Merge taxi data with weather data using temporal join on the hourly timestamp
taxi_weather_df = taxi_df.merge(
    weather_dd,
    on="pickup_time_hour",
    how="left"
)

In [41]:
# Check join quality
missing_weather = taxi_weather_df[
    [
        "temperature_c",
        "precipitation_mm",
        "rain_mm",
        "cloudcover_pct",
        "windspeed_kmh",
    ]
].isna().mean().compute()

print("Missing weather fraction:")
print(missing_weather)

Missing weather fraction:
temperature_c       0.000632
precipitation_mm    0.000632
rain_mm             0.000632
cloudcover_pct      0.000632
windspeed_kmh       0.000632
dtype: float64


After joining hourly weather observations with taxi pickup timestamps, weather information was successfully matched for approximately 99.94% of trips, indicating a highly reliable temporal alignment between the datasets.

In [43]:
taxi_weather_df = taxi_weather_df.persist()
taxi_weather_df = taxi_weather_df.repartition(partition_size="256MB")
taxi_weather_df.to_parquet(
    PARQUET_OUTPUT,
    engine="pyarrow",
    write_index=False
)

In [47]:
taxi_weather_df[[
    "pickup_time_hour",
    "temperature_c",
    "precipitation_mm",
    "rain_mm",
    "cloudcover_pct",
    "windspeed_kmh",
]].drop_duplicates().compute().sort_values("pickup_time_hour").head(10)

,pickup_time_hour,temperature_c,precipitation_mm,rain_mm,cloudcover_pct,windspeed_kmh
0,2018-11-28 15:00:00,4.0,0.0,0.0,100.0,32.9
0,2018-11-28 16:00:00,4.4,0.0,0.0,100.0,31.4
0,2018-11-28 17:00:00,4.9,0.0,0.0,100.0,30.6
0,2018-11-29 10:00:00,4.3,0.0,0.0,91.0,16.9
0,2018-12-21 13:00:00,15.1,1.7,1.7,100.0,21.3
0,2018-12-30 20:00:00,3.5,0.0,0.0,85.0,4.0
0,2018-12-30 21:00:00,3.9,0.0,0.0,86.0,2.9
0,2018-12-31 12:00:00,0.7,0.0,0.0,45.0,4.6
0,2018-12-31 13:00:00,0.8,0.0,0.0,58.0,2.7
0,2018-12-31 14:00:00,1.7,0.0,0.0,61.0,4.9


###  Vicinity of different schools and universities (based on pickup/dropoff location)

Dataset: 
- School Point Locations (obtained from https://data.cityofnewyork.us/Education/School-Point-Locations/jfju-ynrr/about_data)
- Taxi Zones (obtained from https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page)

In [51]:
taxi_weather_df = dd.read_parquet(PARQUET_OUTPUT)

In [52]:
# Convert location IDs to nullable integers to handle missing values and mismatches with taxi zones shapefile
taxi_weather_df["PULocationID"] = taxi_weather_df["PULocationID"].astype("Int64")
taxi_weather_df["DOLocationID"] = taxi_weather_df["DOLocationID"].astype("Int64")

In [37]:
SCHOOLS_PATH = Path("/d/hpc/home/sm79111/bigdata/SchoolPoints_APS_2024_08_28/SchoolPoints_APS_2024_08_28.shp")

schools = gpd.read_file(SCHOOLS_PATH)

schools.head()

,ATS,Building_C,Location_C,Name,Geographic,Latitude,Longitude,geometry
0,01M015,M015,M015,P.S. 015 Roberto Clemente,1,40.722075,-73.978747,POINT (-8235276.446 4971433.816)
1,01M020,M020,M020,P.S. 020 Anna Silver,1,40.721305,-73.986312,POINT (-8236118.578 4971320.718)
2,01M034,M034,M034,P.S. 034 Franklin D. Roosevelt,1,40.726008,-73.975058,POINT (-8234865.788 4972011.521)
3,01M063,M063,M063,The STAR Academy - P.S.63,1,40.724440,-73.986214,POINT (-8236107.668 4971781.199)
4,01M064,M064,M064,P.S. 064 Robert Simon,1,40.723130,-73.981597,POINT (-8235593.706 4971588.778)


In [38]:
schools.crs

<Projected CRS: EPSG:3857>
Name: WGS 84 / Pseudo-Mercator
Axis Info [cartesian]:
- X[east]: Easting (metre)
- Y[north]: Northing (metre)
Area of Use:
- name: World between 85.06°S and 85.06°N.
- bounds: (-180.0, -85.06, 180.0, 85.06)
Coordinate Operation:
- name: Popular Visualisation Pseudo-Mercator
- method: Popular Visualisation Pseudo Mercator
Datum: World Geodetic System 1984 ensemble
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich

In [20]:
print("Number of schools:", len(schools))

Number of schools: 1950


In [39]:
TAXI_ZONES_PATH = Path("/d/hpc/home/sm79111/bigdata/taxi_zones/taxi_zones/taxi_zones.shp")

zones = gpd.read_file(TAXI_ZONES_PATH)

print(zones.head())
print(zones.crs)

   OBJECTID  Shape_Leng  Shape_Area                     zone  LocationID  \
0         1    0.116357    0.000782           Newark Airport           1   
1         2    0.433470    0.004866              Jamaica Bay           2   
2         3    0.084341    0.000314  Allerton/Pelham Gardens           3   
3         4    0.043567    0.000112            Alphabet City           4   
4         5    0.092146    0.000498            Arden Heights           5   

         borough                                           geometry  
0            EWR  POLYGON ((933100.918 192536.086, 933091.011 19...  
1         Queens  MULTIPOLYGON (((1033269.244 172126.008, 103343...  
2          Bronx  POLYGON ((1026308.77 256767.698, 1026495.593 2...  
3      Manhattan  POLYGON ((992073.467 203714.076, 992068.667 20...  
4  Staten Island  POLYGON ((935843.31 144283.336, 936046.565 144...  
EPSG:2263


In [40]:
# Convert both to same CRS (since Taxi zones are natively in 2263)
schools = schools.to_crs("EPSG:2263")
zones = zones.to_crs("EPSG:2263")

In [41]:
schools = schools.dropna(subset=["Latitude", "Longitude"])

schools_gdf = gpd.GeoDataFrame(
    schools,
    geometry=gpd.points_from_xy(schools["Longitude"], schools["Latitude"]),
    crs="EPSG:4326"
)

schools_gdf = schools_gdf.to_crs("EPSG:2263")

In [ ]:
# vicinity ? -> near taxi zones, not just inside
# zones_buffered = zones.copy()
# zones_buffered["geometry"] = zones_buffered.geometry.buffer(500)  # 500 meters

# Spatial Join - Assign each school to a taxi zone
schools_with_zones = gpd.sjoin(
    schools_gdf,
    zones[["LocationID", "zone", "borough", "geometry"]],
    how="inner",
    predicate="within"
)

In [45]:
schools_with_zones.head()

,ATS,Building_C,Location_C,Name,Geographic,Latitude,Longitude,geometry,index_right,LocationID,zone,borough
0,01M015,M015,M015,P.S. 015 Roberto Clemente,1,40.722075,-73.978747,POINT (990141.098 202348.644),3,4,Alphabet City,Manhattan
1,01M020,M020,M020,P.S. 020 Anna Silver,1,40.721305,-73.986312,POINT (988044.207 202067.691),147,148,Lower East Side,Manhattan
2,01M034,M034,M034,P.S. 034 Franklin D. Roosevelt,1,40.726008,-73.975058,POINT (991163.24 203781.827),3,4,Alphabet City,Manhattan
3,01M063,M063,M063,The STAR Academy - P.S.63,1,40.724440,-73.986214,POINT (988071.192 203209.873),78,79,East Village,Manhattan
4,01M064,M064,M064,P.S. 064 Robert Simon,1,40.723130,-73.981597,POINT (989351.029 202732.834),3,4,Alphabet City,Manhattan


In [43]:
# Aggregate Schools per Zone
zone_school_counts = (
    schools_with_zones.groupby("LocationID")
    .size()
    .reset_index(name="school_count")
)

In [46]:
zone_school_counts.head()

,LocationID,school_count
0,3,6
1,4,7
2,5,2
3,6,4
4,7,9


In [47]:
zone_school_counts["LocationID"] = zone_school_counts["LocationID"].astype("Int64")

In [48]:
zone_school_counts_dd = dd.from_pandas(zone_school_counts, npartitions=1)

In [53]:
# Join with taxi data to get school counts per zone

# Pickup
taxi_augmented_df = taxi_weather_df.merge(
    zone_school_counts_dd.rename(columns={
        "LocationID": "PULocationID",
        "school_count": "pickup_school_count"
    }),
    on="PULocationID",
    how="left"
)

# Dropoff
taxi_augmented_df = taxi_augmented_df.merge(
    zone_school_counts_dd.rename(columns={
        "LocationID": "DOLocationID",
        "school_count": "dropoff_school_count"
    }),
    on="DOLocationID",
    how="left"
)

In [55]:
# zones without schools will have NaN, fill with 0
taxi_augmented_df["pickup_school_count"] = (
    taxi_augmented_df["pickup_school_count"].fillna(0)
)

taxi_augmented_df["dropoff_school_count"] = (
    taxi_augmented_df["dropoff_school_count"].fillna(0)
)

In [57]:
taxi_augmented_df[[
    "PULocationID",
    "DOLocationID",
    "pickup_school_count",
    "dropoff_school_count"
]].head(20)

,PULocationID,DOLocationID,pickup_school_count,dropoff_school_count
0,138,186,0.0,3.0
1,162,161,0.0,0.0
2,142,239,1.0,12.0
3,87,249,0.0,5.0
4,161,230,0.0,2.0
5,13,107,5.0,16.0
6,50,114,7.0,0.0
7,132,132,0.0,0.0
8,137,107,1.0,16.0
9,132,80,0.0,8.0


In [58]:
taxi_augmented_df = taxi_augmented_df.persist()
taxi_augmented_df = taxi_augmented_df.repartition(partition_size="256MB")
taxi_augmented_df.to_parquet(
    OUTPUT_DIR / "augmented_weather_schools_parquet",
    engine="pyarrow",
    write_index=False
)

### Vicinity of major businesses and attractions (based on pickup/dropoff date-time)

Dataset: 

In [70]:
taxi_augmented_df = dd.read_parquet(OUTPUT_DIR / "augmented_weather_schools_parquet")

In [59]:
taxi_zones = gpd.read_file(TAXI_ZONES_PATH)
taxi_zones = taxi_zones[["LocationID", "zone", "borough", "geometry"]]

In [60]:
# load business csv 
businesses = pd.read_csv("/d/hpc/home/sm79111/bigdata/businesses_20260406.csv")
businesses.head()

,Business Name,Community Board,Council District,BIN,BBL,Census Tract (2010),Latitude,Longitude
0,"GREGORY ULTO, INC.",318.0,45.0,3217486.0,3.078250e+09,730.0,40.62446,-73.932304
1,YA.K.K INC,103.0,1.0,1001812.0,1.001640e+09,29.0,40.71573,-73.998788
2,"PROCESS SERVER PLUS, INC.",409.0,32.0,4188647.0,4.090710e+09,38.0,40.68463,-73.844767
3,BROADWAY BEER AND SMOKE INC.,401.0,22.0,4008258.0,4.006110e+09,59.0,40.76188,-73.925212
4,NEW YORK GINSENG CITY INC,407.0,20.0,4112315.0,4.049760e+09,871.0,40.75913,-73.831422


In [61]:
businesses = businesses.dropna(subset=["Longitude", "Latitude"])

In [62]:
businesses_gdf = gpd.GeoDataFrame(
    businesses,
    geometry=gpd.points_from_xy(businesses["Longitude"], businesses["Latitude"]),
    crs="EPSG:4326"
)
businesses_gdf = businesses_gdf.to_crs(taxi_zones.crs)

In [63]:
businesses_with_zones = gpd.sjoin(
    businesses_gdf,
    taxi_zones,
    how="inner",
    predicate="within"
)
zone_business_counts = (
    businesses_with_zones.groupby("LocationID")
    .size()
    .reset_index(name="business_count")
)

In [64]:
zone_business_counts["LocationID"] = zone_business_counts["LocationID"].astype("Int64")

In [65]:
zone_business_counts_dd = dd.from_pandas(zone_business_counts, npartitions=1)

In [71]:
# Ensure taxi IDs are correct
taxi_augmented_df["PULocationID"] = taxi_augmented_df["PULocationID"].astype("Int64")
taxi_augmented_df["DOLocationID"] = taxi_augmented_df["DOLocationID"].astype("Int64")

# Pickup
taxi_augmented_df = taxi_augmented_df.merge(
    zone_business_counts_dd.rename(columns={
        "LocationID": "PULocationID",
        "business_count": "pickup_business_count"
    }),
    on="PULocationID",
    how="left"
)

# Dropoff
taxi_augmented_df = taxi_augmented_df.merge(
    zone_business_counts_dd.rename(columns={
        "LocationID": "DOLocationID",
        "business_count": "dropoff_business_count"
    }),
    on="DOLocationID",
    how="left"
)

taxi_augmented_df["pickup_business_count"] = taxi_augmented_df["pickup_business_count"].fillna(0)
taxi_augmented_df["dropoff_business_count"] = taxi_augmented_df["dropoff_business_count"].fillna(0)

In [72]:
# Create a combined activity index as a simple sum of nearby businesses and schools
taxi_augmented_df["activity_index"] = (
    taxi_augmented_df["pickup_business_count"] +
    taxi_augmented_df["pickup_school_count"] +
    taxi_augmented_df["dropoff_business_count"] +
    taxi_augmented_df["dropoff_school_count"]
)

In [73]:
taxi_augmented_df.head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,...,temperature_c,precipitation_mm,rain_mm,cloudcover_pct,windspeed_kmh,pickup_school_count,dropoff_school_count,pickup_business_count,dropoff_business_count,activity_index
0,2.0,2019-01-01 14:00:38,2019-01-01 14:21:10,2.0,8.97,1.0,N,138,186,1.0,...,13.2,0.0,0.0,86.0,27.0,0,3,3.0,178.0,184.0
1,2.0,2019-01-01 14:00:34,2019-01-01 14:03:00,1.0,0.46,1.0,N,162,161,2.0,...,13.2,0.0,0.0,86.0,27.0,0,0,190.0,562.0,752.0
2,2.0,2019-01-01 14:00:00,2019-01-01 14:05:28,2.0,1.10,1.0,N,142,239,1.0,...,13.2,0.0,0.0,86.0,27.0,1,12,141.0,163.0,317.0
3,2.0,2019-01-01 14:02:42,2019-01-01 14:14:22,1.0,2.85,1.0,N,87,249,2.0,...,13.2,0.0,0.0,86.0,27.0,0,5,203.0,154.0,362.0
4,2.0,2019-01-01 14:01:08,2019-01-01 14:05:31,3.0,0.57,1.0,N,161,230,2.0,...,13.2,0.0,0.0,86.0,27.0,0,2,562.0,248.0,812.0


In [74]:
taxi_augmented_df.to_parquet(
    OUTPUT_DIR / "augmented_weather_schools_businesses_parquet",
    engine="pyarrow",
    write_index=False
)

### Major events in the vicinity (based on pickup/dropoff date-time and location)

Dataset

In [23]:
client.close()
cluster.close()

/d/hpc/home/sm79111/.conda/envs/bd39/lib/python3.9/site-packages/dask_jobqueue/core.py:266: FutureWarning: job_extra has been renamed to job_extra_directives. You are still using it (even if only set to []; please also check config files). If you did not set job_extra_directives yet, job_extra will be respected for now, but it will be removed in a future release. If you already set job_extra_directives, job_extra is ignored and you can remove it.
  warnings.warn(warn, FutureWarning)
